# 00 — Project Overview

Този notebook дава ясен преглед на Lottery Probability Model: данни, модели, отчети, Streamlit UI и крайния процес за изграждане на фишове.

Проектът не е система за гарантирана печалба. Той е аналитична система, която комбинира статистически сигнали, backtest резултати, model weighting и portfolio construction.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Основни файлове

- `data/historical_draws.csv` — основен исторически dataset.
- `data/v40_normalized_draw_events.csv` — нормализирани draw events.
- `data/v41_canonical_draw_events.csv` — canonical draw events за model layer.
- `reports/*.csv` — резултати от модели, backtests, ticket builders, audits и dashboard snapshots.
- `streamlit_app.py` + `app.py` — потребителският UI.


In [ ]:
files = [
    DATA_DIR / "historical_draws.csv",
    DATA_DIR / "v40_normalized_draw_events.csv",
    DATA_DIR / "v41_canonical_draw_events.csv",
    REPORTS_DIR / "v117_real_ticket_pack_builder_cards.csv",
    REPORTS_DIR / "v118_model_system_ticket_pack.csv",
    REPORTS_DIR / "v118_full_system_price_table.csv",
    PROJECT_ROOT / "streamlit_app.py",
    PROJECT_ROOT / "app.py",
]
file_status(files)


## Общ pipeline

1. Исторически данни.
2. Нормализация и canonical event layer.
3. Статистически модели — frequency, gap, balance, pair/group logic.
4. ML extensions — classification, clustering, dimensionality reduction, meta learner.
5. Backtest и сравнение.
6. Ensemble / weighting.
7. Ticket builder и portfolio construction.
8. Streamlit UI за реална употреба.


In [ ]:
report_files = sorted(REPORTS_DIR.glob("*.csv"))
model_dirs = sorted([p for p in MODELS_DIR.glob("v*") if p.is_dir()])
pd.DataFrame({
    "metric": ["CSV report files", "Model/version folders", "Draw rows"],
    "value": [len(report_files), len(model_dirs), len(draws)],
})
